# Номера договоров ТЭ: Снигур, отчёт за август, final_df и моя выгрузка

Гипотеза: клиентов у меня примерно столько же, сколько у Снигура, но номер договора я подтягиваю не тот, поэтому сверка по номеру даёт большое расхождение.

Источники:
- **Снигур** - `cdwh_acq_clients.xlsx` (ЦФТ / CDWH), ключ - номер договора, ИНН нет;
- **Отчёт за август** - `Data/08_Август_2026.xlsx` (6656 договоров), колонки находятся автоматически: ID договора, номер договора, ИНН, наименование;
- **final_df за август** - `Data/final_df_period_2026_01_2026_08_final_df_script_3_1.csv`, `report_month = 2026-08`. `agr_id` = `abs_agr_id` из `ods_alpha` = `id` мерчанта в ЦФТ (`ods.scd1_z_r2_ip_merchants`), `contract_number` = `c_agr_number` из `ods_alpha`;
- **Моя выгрузка** - лист `С ТЭ на 01.09.2026`, номер тоже из `c_agr_number`.

Что считается:
1. совпадение ключей между всеми парами источников: по номеру, по цифрам номера, по ID договора, по ИНН, по наименованию. Если по ИНН и наименованию совпадение высокое, а по номеру низкое - гипотеза подтверждается;
2. final_df и отчёт по ID договора: совпадают ли номер, ИНН и наименование;
3. если номер в отчёте и в final_df разный - какой из них есть у Снигура, и сколько договоров моей выгрузки нашлось бы у Снигура, если брать номер из отчёта;
4. в какой колонке `ods.scd1_z_r2_ip_merchants` лежит номер в формате Снигура (нужен логин Impala, `LOOKUP_CFT`).

Снигур - снимок на 01.09, отчёт и final_df - весь август, поэтому полного совпадения не будет даже при одинаковых номерах. Смотреть надо на разницу между «по номеру» и «по клиенту».


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 80)

REPORT_MONTH = '2026-08'
AS_OF_DATE = '2026-09-01'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
DATA_DIR = BASE_DIR / 'Data'
EX_FILE = DATA_DIR / '08_Август_2026.xlsx'
EX_SHEET = 0
FINAL_DF_CSV = DATA_DIR / 'final_df_period_2026_01_2026_08_final_df_script_3_1.csv'
SNIGUR_FILE = BASE_DIR / 'Скрипт для Ани по сборке продукта эквайринг' / 'cdwh_acq_clients.xlsx'
SNIGUR_SHEET = 0
MINE_FILE = BASE_DIR / ('trade_acquiring_clients_%s.xlsx' % AS_OF_DATE)
OUT_FILE = BASE_DIR / ('te_numbers_aug_report_check_%s.xlsx' % REPORT_MONTH)

# Колонки отчёта. None - найти автоматически, иначе точное имя колонки
EX_ID_COL = None
EX_NUM_COL = None
EX_INN_COL = None
EX_NAME_COL = None

LOOKUP_CFT = True   # искать номер Снигура в ods.scd1_z_r2_ip_merchants
CFT_SAMPLE = 300    # договоров final_df в выборке для поиска колонки
CFT_NUM_COL = None  # если колонка уже известна - указать имя

as_of = pd.Timestamp(AS_OF_DATE)
as_of_ru = as_of.strftime('%d.%m.%Y')
MINE_SHEET = 'С ТЭ на %s' % as_of_ru
for p in (EX_FILE, FINAL_DF_CSV, SNIGUR_FILE, MINE_FILE):
    print('есть ' if p.exists() else 'НЕТ  ', p)


## 1. Загрузка

In [ ]:
LEGAL_FORMS = [
    'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ', 'НЕПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО',
    'АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ИНДИВИДУАЛЬНЫЙ ПРЕДПРИНИМАТЕЛЬ', 'ГЛАВА КФХ',
    'ООО', 'ОАО', 'ЗАО', 'ПАО', 'НАО', 'АО', 'ИП', 'МБУ', 'МАУ', 'МБОУ', 'МАОУ', 'МБДОУ', 'МАДОУ', 'МКУ', 'ГБУ',
    'ГАУ', 'ГКУ', 'ГБУЗ', 'ФГБУ', 'ФГУП', 'МУП', 'ГУП', 'АНО', 'ТСЖ', 'СНТ', 'КФХ',
]
LEGAL_RE = re.compile(r'\b(%s)\b' % '|'.join(LEGAL_FORMS))
EMPTY = ('', 'NONE', 'NAN', '<NA>', 'NAT', 'NULL')

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def norm_agr(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.')[0]
    s = re.sub(r'\s+', '', s).upper()
    return None if s in EMPTY else s

def digits_key(v):
    k = norm_agr(v)
    if not k:
        return None
    d = re.sub(r'\D', '', k).lstrip('0')
    return d if len(d) >= 4 else None

def norm_id(v):
    k = norm_agr(v)
    if not k:
        return None
    return (k.lstrip('0') or '0') if k.isdigit() else k

def norm_inn(v):
    if v is None or pd.isna(v):
        return None
    s = re.sub(r'\D', '', str(v).split('.')[0])
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None

def norm_name(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).upper().replace('Ё', 'Е')
    s = re.sub(r'[^0-9A-ZА-Я]+', ' ', s)
    s = LEGAL_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return None if s in EMPTY else s

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    return out.dt.normalize()

def read_with_header(path, sheet, need=('договор',), max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        cells = ' | '.join(_low(v) for v in raw.iloc[i] if pd.notna(v))
        if all(k in cells for k in need):
            names, seen = [], {}
            for j, v in enumerate(raw.iloc[i]):
                n = re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else 'col_%s' % j
                seen[n] = seen.get(n, 0) + 1
                names.append(n if seen[n] == 1 else '%s_%s' % (n, seen[n]))
            df = raw.iloc[i + 1:].copy()
            df.columns = names
            return df.dropna(how='all').reset_index(drop=True), i
    raise ValueError('Не нашёл строку заголовков с %s в первых %s строках %s' % (need, max_rows, path))

def pick(cols, exact, contains=None, exclude=()):
    low = {c: _low(c) for c in cols}
    for e in exact:
        for c in cols:
            if low[c] == e:
                return c
    if contains:
        for c in cols:
            if all(k in low[c] for k in contains) and not any(x in low[c] for x in exclude):
                return c
    return None

def digits_share(s):
    v = s.dropna().map(lambda x: re.sub(r'\.0+$', '', str(x).strip()))
    v = v[v != '']
    return float(v.str.fullmatch(r'\d{6,}').mean()) if len(v) else 0.0

def col_or_none(df, col, fn=None):
    if col is None:
        return pd.Series(None, index=df.index, dtype=object)
    s = df[col]
    return s.map(fn) if fn else s.astype('string').str.strip()


In [ ]:
rep_raw, rep_hdr = read_with_header(EX_FILE, EX_SHEET, need=('инн', 'договор'))
cols = list(rep_raw.columns)
print('Отчёт: строка заголовка', rep_hdr, '| строк', len(rep_raw))
print('Колонки отчёта:', cols)

ex_id = EX_ID_COL or pick(cols, ['id договора', 'agr_id', 'abs_agr_id'], ('id', 'договор'))
ex_num = EX_NUM_COL or pick(cols, ['номер договора', '№ договора'], ('номер', 'договор'), exclude=('id',))
ex_inn = EX_INN_COL or pick(cols, ['инн', 'инн клиента'], ('инн',), exclude=('банк',))
ex_name = EX_NAME_COL or pick(
    cols,
    ['наименование клиента', 'наименование организации', 'наименование', 'название организации',
     'клиент', 'организация', 'наименование тсп'],
    ('наимен',), exclude=('тариф', 'филиал', 'всп', 'отдел', 'рф', 'mcc', 'мсс', 'банк'))

for c in (ex_id, ex_num):
    if c:
        print('%s: доля значений из одних цифр (6+) = %.3f' % (c, digits_share(rep_raw[c])))
if ex_num and not EX_NUM_COL and digits_share(rep_raw[ex_num]) > 0.95:
    if ex_id is None:
        print('«%s» содержит только цифры - это ID договора, а не номер' % ex_num)
        ex_id, ex_num = ex_num, None
    else:
        print('ВНИМАНИЕ: «%s» тоже из одних цифр - проверить, номер ли это' % ex_num)
if ex_id is None and ex_num is None:
    raise KeyError('В отчёте нет ни ID, ни номера договора. Задайте EX_ID_COL / EX_NUM_COL')
print('Колонки отчёта для сверки: ID = %r | номер = %r | ИНН = %r | наименование = %r' % (ex_id, ex_num, ex_inn, ex_name))
display(rep_raw[[c for c in (ex_id, ex_num, ex_inn, ex_name) if c]].head(10))

rep = pd.DataFrame({
    'id': col_or_none(rep_raw, ex_id, norm_id),
    'ID договора (отчёт)': col_or_none(rep_raw, ex_id),
    'num_key': col_or_none(rep_raw, ex_num, norm_agr),
    'Номер договора (отчёт)': col_or_none(rep_raw, ex_num),
    'inn_n': col_or_none(rep_raw, ex_inn, norm_inn),
    'ИНН (отчёт)': col_or_none(rep_raw, ex_inn),
    'Наименование (отчёт)': col_or_none(rep_raw, ex_name),
})
rep['digits'] = rep['num_key'].map(digits_key)
rep['name_n'] = rep['Наименование (отчёт)'].map(norm_name)
rep_key = 'id' if ex_id else 'num_key'
rep = rep[rep[rep_key].notna()]
rep_rows = len(rep)
rep = rep.drop_duplicates(rep_key).reset_index(drop=True)
print('Отчёт: строк с договором %s, уникальных по %s %s' % (rep_rows, rep_key, len(rep)))


In [ ]:
fd_need = ['report_month', 'agr_id', 'n_agr', 'contract_number', 'contract_number_acq', 'inn', 'company_name',
           'd_valid_from', 'd_valid_to']
fd_all = pd.read_csv(FINAL_DF_CSV, dtype=str, low_memory=False, usecols=lambda c: c in fd_need)
fd_month = fd_all['report_month'].astype('string').str.strip().str[:7]
print('final_df: месяцы', sorted(fd_month.dropna().unique()))
fd_m = fd_all[fd_month == REPORT_MONTH].copy()
if fd_m.empty:
    raise ValueError('В final_df нет месяца %s' % REPORT_MONTH)
for c in fd_need:
    if c not in fd_m.columns:
        fd_m[c] = None
fd_m['id'] = fd_m['agr_id'].map(norm_id)
fd_m['num_key'] = fd_m['contract_number'].map(norm_agr)
fd_nums_per_id = fd_m.dropna(subset=['id']).groupby('id')['num_key'].nunique()
fd = pd.DataFrame({
    'id': fd_m['id'],
    'agr_id (final_df)': fd_m['agr_id'],
    'n_agr (final_df)': fd_m['n_agr'],
    'num_key': fd_m['num_key'],
    'Номер договора (final_df)': fd_m['contract_number'],
    'Номер актуальный (final_df)': fd_m['contract_number_acq'],
    'num_acq_key': fd_m['contract_number_acq'].map(norm_agr),
    'inn_n': fd_m['inn'].map(norm_inn),
    'ИНН (final_df)': fd_m['inn'],
    'Наименование (final_df)': fd_m['company_name'],
    'Начало (final_df)': to_date(fd_m['d_valid_from']),
    'Окончание (final_df)': to_date(fd_m['d_valid_to']),
})
fd['digits'] = fd['num_key'].map(digits_key)
fd['name_n'] = fd['Наименование (final_df)'].map(norm_name)
fd_rows = len(fd)
fd = fd.dropna(subset=['id']).drop_duplicates('id').reset_index(drop=True)
print('final_df %s: строк %s, договоров (agr_id) %s, agr_id с 2+ номерами %s' % (
    REPORT_MONTH, fd_rows, len(fd), int((fd_nums_per_id > 1).sum())))
acq_diff = fd['num_acq_key'].notna() & (fd['num_acq_key'] != fd['num_key'])
print('contract_number_acq заполнен у %s, отличается от contract_number у %s' % (
    int(fd['num_acq_key'].notna().sum()), int(acq_diff.sum())))


In [ ]:
sn_raw, _ = read_with_header(SNIGUR_FILE, SNIGUR_SHEET)
sn_cols = list(sn_raw.columns)
c_name = pick(sn_cols, ['наименование клиента'], ('наимен',))
c_agr = pick(sn_cols, ['номер договора'], ('номер', 'договор'))
c_start = pick(sn_cols, ['дата начала действия договора'], ('начал',))
c_end = pick(sn_cols, ['дата окончания действия договора'], ('окончан',))
c_close = pick(sn_cols, ['дата закрытия'], ('закрыт', 'дата'))
c_status = pick(sn_cols, ['статус'], ('статус',))
print('Колонки Снигура:', c_name, c_agr, c_start, c_end, c_close, c_status, sep=' | ')

sn = pd.DataFrame({
    'num_key': sn_raw[c_agr].map(norm_agr),
    'Номер договора (Снигур)': sn_raw[c_agr].astype('string').str.strip(),
    'Клиент (Снигур)': sn_raw[c_name].astype('string').str.strip(),
    'Дата начала (Снигур)': to_date(sn_raw[c_start]),
    'Конец (Снигур)': pd.concat([to_date(sn_raw[c_end]), to_date(sn_raw[c_close])], axis=1).min(axis=1),
    'Статус (Снигур)': col_or_none(sn_raw, c_status),
})
sn = sn[sn['num_key'].notna()]
sn['active'] = (sn['Дата начала (Снигур)'].isna() | (sn['Дата начала (Снигур)'] <= as_of)) & (
    sn['Конец (Снигур)'].isna() | (sn['Конец (Снигур)'] >= as_of))
sn = sn.sort_values('active', ascending=False).drop_duplicates('num_key').reset_index(drop=True)
sn['digits'] = sn['num_key'].map(digits_key)
sn['name_n'] = sn['Клиент (Снигур)'].map(norm_name)
sn['id'] = None
sn['inn_n'] = None
print('Снигур: договоров %s, действуют на %s (по более ранней из дат окончания и закрытия) %s' % (
    len(sn), as_of_ru, int(sn['active'].sum())))

mine_raw = pd.read_excel(MINE_FILE, sheet_name=MINE_SHEET, dtype=object)
mine = pd.DataFrame({
    'num_key': mine_raw['Номер договора'].map(norm_agr),
    'Номер договора (мой)': mine_raw['Номер договора'].astype('string'),
    'inn_n': mine_raw['ИНН'].map(norm_inn),
    'ЦФТ ID': mine_raw['ЦФТ ID'].astype('string'),
})
mine = mine[mine['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
mine['digits'] = mine['num_key'].map(digits_key)
fd_by_num = fd.dropna(subset=['num_key']).drop_duplicates('num_key').set_index('num_key')
mine['id'] = mine['num_key'].map(fd_by_num['id'])
mine['name_n'] = mine['num_key'].map(fd_by_num['name_n'])
print('Моя выгрузка: договоров %s, из них нашлись в final_df по номеру %s' % (len(mine), int(mine['id'].notna().sum())))

loaded = pd.DataFrame([
    ('Отчёт за август: строк с договором', rep_rows),
    ('Отчёт за август: уникальных договоров (%s)' % rep_key, len(rep)),
    ('final_df за август: договоров (agr_id)', len(fd)),
    ('Снигур: договоров', len(sn)),
    ('Снигур: действуют на %s' % as_of_ru, int(sn['active'].sum())),
    ('Моя выгрузка на %s: договоров' % as_of_ru, len(mine)),
], columns=['Загружено', 'Значение'])
display(loaded)


## 2. Совпадение ключей между источниками

Для каждой пары источников и каждого ключа: сколько ключей у каждого, сколько общих, сколько только у одного. «% A найдено в B» - доля ключей первого источника, которые есть у второго.

- **номер** - номер договора без пробелов, в верхнем регистре;
- **цифры номера** - только цифры номера без ведущих нулей (ловит разную запись: `ИЭД_6300/41` и `6300-41`);
- **ID договора** - `agr_id`; у моей выгрузки он взят из final_df по номеру;
- **ИНН**, **наименование** - уровень клиента. У Снигура ИНН нет. Наименование моей выгрузки взято из final_df по номеру.


In [ ]:
sources = {
    'Снигур, все': sn,
    'Снигур, действуют на %s' % as_of_ru: sn[sn['active']],
    'Отчёт за август': rep,
    'final_df за август': fd,
    'Моя выгрузка на %s' % as_of_ru: mine,
}
SN_ACT, REP, FD, MINE = list(sources)[1:]
pairs = [
    ('Снигур, все', REP), (SN_ACT, REP), (FD, REP), (MINE, REP),
    (SN_ACT, FD), (SN_ACT, MINE), (MINE, FD),
]
key_kinds = [('номер', 'num_key'), ('цифры номера', 'digits'), ('ID договора', 'id'),
             ('ИНН', 'inn_n'), ('наименование', 'name_n')]

def keyset(df, col):
    return set(df[col].dropna()) if col in df.columns else set()

rows = []
for a, b in pairs:
    for kind, col in key_kinds:
        ka, kb = keyset(sources[a], col), keyset(sources[b], col)
        if not ka or not kb:
            continue
        common = len(ka & kb)
        rows.append({
            'A': a, 'B': b, 'Ключ': kind, 'Ключей у A': len(ka), 'Ключей у B': len(kb), 'Общих': common,
            'Только у A': len(ka - kb), 'Только у B': len(kb - ka),
            '% A найдено в B': round(common / len(ka) * 100, 1), '% B найдено в A': round(common / len(kb) * 100, 1),
        })
overlap = pd.DataFrame(rows)
display(overlap)


## 3. final_df и отчёт за август по ID договора

Связь по `agr_id` / «ID договора». Для договоров, которые есть в обоих, сравниваются номер, ИНН и наименование. Для договоров только с одной стороны - есть ли там тот же номер или тот же ИНН.


In [ ]:
def cmp_text(a, b, digits=False):
    if a is None or pd.isna(a):
        return 'в final_df пусто'
    if b is None or pd.isna(b):
        return 'в отчёте пусто'
    if a == b:
        return 'совпадает'
    if digits and digits_key(a) and digits_key(a) == digits_key(b):
        return 'совпадает по цифрам'
    if not digits and (a in b or b in a):
        return 'похоже (одно содержит другое)'
    return 'отличается'

fd_rep = pd.DataFrame()
fd_rep_tables = []
if ex_id:
    fd_rep = fd.merge(rep, on='id', how='outer', suffixes=(' fd', ' rep'), indicator=True)
    fd_rep['Где есть'] = fd_rep['_merge'].map({'both': 'в обоих', 'left_only': 'только в final_df', 'right_only': 'только в отчёте'})
    both = fd_rep['_merge'] == 'both'
    for col in ('Номер', 'Номер актуальный', 'ИНН', 'Наименование', 'Почему одна сторона'):
        fd_rep[col] = pd.Series(pd.NA, index=fd_rep.index, dtype=object)
    fd_rep.loc[both, 'Номер'] = [cmp_text(a, b, True) for a, b in zip(fd_rep.loc[both, 'num_key fd'], fd_rep.loc[both, 'num_key rep'])]
    fd_rep.loc[both, 'Номер актуальный'] = [cmp_text(a, b, True) for a, b in zip(fd_rep.loc[both, 'num_acq_key'], fd_rep.loc[both, 'num_key rep'])]
    fd_rep.loc[both, 'ИНН'] = [cmp_text(a, b) for a, b in zip(fd_rep.loc[both, 'inn_n fd'], fd_rep.loc[both, 'inn_n rep'])]
    fd_rep.loc[both, 'Наименование'] = [cmp_text(a, b) for a, b in zip(fd_rep.loc[both, 'name_n fd'], fd_rep.loc[both, 'name_n rep'])]

    fd_nums, fd_inns = keyset(fd, 'num_key'), keyset(fd, 'inn_n')
    rep_nums, rep_inns = keyset(rep, 'num_key'), keyset(rep, 'inn_n')
    def side_reason(num, inn, nums, inns, other):
        if num and num in nums:
            return 'номер есть в %s под другим ID' % other
        if inn and inn in inns:
            return 'ИНН есть в %s, договор другой' % other
        return 'ни номера, ни ИНН в %s нет' % other
    only_rep = fd_rep['_merge'] == 'right_only'
    fd_rep.loc[only_rep, 'Почему одна сторона'] = [
        side_reason(n, i, fd_nums, fd_inns, 'final_df')
        for n, i in zip(fd_rep.loc[only_rep, 'num_key rep'], fd_rep.loc[only_rep, 'inn_n rep'])]
    only_fd = fd_rep['_merge'] == 'left_only'
    fd_rep.loc[only_fd, 'Почему одна сторона'] = [
        side_reason(n, i, rep_nums, rep_inns, 'отчёте')
        for n, i in zip(fd_rep.loc[only_fd, 'num_key fd'], fd_rep.loc[only_fd, 'inn_n fd'])]

    where = fd_rep['Где есть'].value_counts().rename_axis('final_df и отчёт по ID').reset_index(name='Договоров')
    fd_rep_tables.append(where)
    for col in ('Номер', 'Номер актуальный', 'ИНН', 'Наименование'):
        fd_rep_tables.append(fd_rep.loc[both, col].value_counts()
                             .rename_axis('В обоих по ID: %s' % col.lower()).reset_index(name='Договоров'))
    fd_rep_tables.append(fd_rep['Почему одна сторона'].dropna().value_counts()
                         .rename_axis('Только с одной стороны').reset_index(name='Договоров'))
    for t in fd_rep_tables:
        display(t)
else:
    print('В отчёте нет ID договора - сверка по ID пропущена')

fd_rep_cols = ['Где есть', 'Номер', 'Номер актуальный', 'ИНН', 'Наименование', 'Почему одна сторона',
               'agr_id (final_df)', 'ID договора (отчёт)', 'Номер договора (final_df)', 'Номер актуальный (final_df)',
               'Номер договора (отчёт)', 'ИНН (final_df)', 'ИНН (отчёт)', 'Наименование (final_df)',
               'Наименование (отчёт)', 'n_agr (final_df)', 'Начало (final_df)', 'Окончание (final_df)']
fd_rep_out = fd_rep.reindex(columns=fd_rep_cols) if len(fd_rep) else pd.DataFrame(columns=fd_rep_cols)
num_diff_out = fd_rep_out[fd_rep_out['Номер'].isin(['отличается', 'совпадает по цифрам'])]
name_diff_out = fd_rep_out[fd_rep_out['Наименование'] == 'отличается']
display(num_diff_out.head(15))
display(name_diff_out.head(15))


## 4. Проверка гипотезы: какой номер есть у Снигура

1. Для договоров, у которых номер в final_df и в отчёте разный: чей номер есть у Снигура.
2. Подмена номера в моей выгрузке: мой номер → `agr_id` (через final_df) → номер из отчёта. Сколько договоров совпало бы со Снигуром до и после подмены.


In [ ]:
sn_all_keys = keyset(sn, 'num_key')
sn_act_keys = keyset(sn[sn['active']], 'num_key')
hyp_tables = []

if ex_id and ex_num:
    diff = fd_rep[fd_rep['Номер'].isin(['отличается', 'совпадает по цифрам'])]
    whose = pd.DataFrame({
        'Номер final_df у Снигура': diff['num_key fd'].isin(sn_all_keys).map({True: 'есть', False: 'нет'}),
        'Номер отчёта у Снигура': diff['num_key rep'].isin(sn_all_keys).map({True: 'есть', False: 'нет'}),
    })
    whose_t = (whose.value_counts().rename('Договоров').reset_index()
               .rename(columns={'Номер final_df у Снигура': 'Номер разный: номер final_df у Снигура'}))
    hyp_tables.append(whose_t)
    display(whose_t)

    rep_num_by_id = rep.dropna(subset=['id', 'num_key']).set_index('id')['num_key']
    swapped = mine['id'].map(rep_num_by_id)
    mine_swap = swapped.fillna(mine['num_key'])
    swap = pd.DataFrame([
        ('Моих договоров', len(mine)),
        ('  нашлись в final_df по номеру (есть agr_id)', int(mine['id'].notna().sum())),
        ('  у этого agr_id есть номер в отчёте', int(swapped.notna().sum())),
        ('  номер в отчёте отличается от моего', int((swapped.notna() & (swapped != mine['num_key'])).sum())),
        ('Совпали со Снигуром (действующие) по моему номеру', int(mine['num_key'].isin(sn_act_keys).sum())),
        ('Совпали со Снигуром (действующие) по номеру из отчёта', int(mine_swap.isin(sn_act_keys).sum())),
        ('Совпали со Снигуром (все) по моему номеру', int(mine['num_key'].isin(sn_all_keys).sum())),
        ('Совпали со Снигуром (все) по номеру из отчёта', int(mine_swap.isin(sn_all_keys).sum())),
    ], columns=['Подмена номера в моей выгрузке', 'Договоров'])
    hyp_tables.append(swap)
    display(swap)
else:
    print('Нужны и ID, и номер в отчёте - проверка пропущена. ID: %r, номер: %r' % (ex_id, ex_num))


## 5. Где в ЦФТ лежит номер Снигура

`agr_id` из final_df = `id` в `ods.scd1_z_r2_ip_merchants`. Берём выборку договоров final_df за август:
- **номер совпал** - `contract_number` есть у Снигура;
- **клиент совпал, номер нет** - номера у Снигура нет, но у Снигура есть клиент с тем же наименованием.

По ним читаем все колонки мерчанта и смотрим, в какой колонке значение равно номеру Снигура (для второй группы - любому номеру этого клиента у Снигура). Если такая колонка нашлась, читаем её для всех договоров final_df и пересчитываем совпадение со Снигуром.


In [ ]:
cft_cols_t = pd.DataFrame()
cft_match = pd.DataFrame()
cft_examples = pd.DataFrame()

def sql_in(vals):
    return ', '.join("'%s'" % str(v).replace("'", "''") for v in vals)

if LOOKUP_CFT and fd['num_key'].isin(sn_all_keys).sum() + fd['name_n'].isin(set(sn['name_n'].dropna())).sum() == 0:
    print('Нет договоров final_df, совпавших со Снигуром по номеру или наименованию - поиск колонки пропущен')
elif LOOKUP_CFT:
    from rail_connectors.connection import connect
    if globals().get('imp') is None:
        LAKE_USER = input('Логин Impala: ').strip()
        LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')
        imp = connect(
            to='IMPALA',
            extra_options={'db': 'sandbox_ai'},
            driver_args={'tez.queue.name': 'ai'},
            kerberos={
                'keytab_path': '/home/jovyan/test_requests/tech.keytab',
                'use_credentials': True,
                'update_keytab': True,
            },
            user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
        )
        imp._init_connection()

    sn_nums_by_name = sn.dropna(subset=['name_n']).groupby('name_n')['num_key'].apply(set).to_dict()
    fd_ok = fd[fd['num_key'].isin(sn_all_keys)]
    fd_name = fd[~fd['num_key'].isin(sn_all_keys) & fd['name_n'].isin(sn_nums_by_name)]
    half = CFT_SAMPLE // 2
    sample = pd.concat([
        fd_ok.sample(min(half, len(fd_ok)), random_state=1).assign(group='номер совпал'),
        fd_name.sample(min(half, len(fd_name)), random_state=1).assign(group='клиент совпал, номер нет'),
    ], ignore_index=True)
    print('Выборка:', sample['group'].value_counts().to_dict())

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        merch = imp.fetch(
            "SELECT * FROM ods.scd1_z_r2_ip_merchants m WHERE cast(m.id AS string) IN (%s) "
            "AND coalesce(m.ods_deleted_flg, '0') <> '1'" % sql_in(sample['id']))
    merch.columns = [str(c).split('.')[-1].lower() for c in merch.columns]
    merch['id_key'] = merch['id'].map(norm_id)
    print('Строк мерчантов:', len(merch), '| колонок:', merch.shape[1])
    s = sample.merge(merch, left_on='id', right_on='id_key', how='inner', suffixes=('', ' cft'))

    targets = [({r['num_key']} if g == 'номер совпал' else sn_nums_by_name.get(r['name_n'], set()))
               for g, r in zip(s['group'], s.to_dict('records'))]
    target_digits = [{digits_key(t) for t in ts} - {None} for ts in targets]
    rows = []
    for c in merch.columns:
        if c in ('id_key',):
            continue
        vals = s[c + ' cft'] if c + ' cft' in s.columns else s[c]
        keys = vals.map(norm_agr)
        dig = vals.map(digits_key)
        hit = pd.Series([k in t for k, t in zip(keys, targets)], index=s.index)
        hit_d = pd.Series([d is not None and d in t for d, t in zip(dig, target_digits)], index=s.index)
        if not (hit.any() or hit_d.any()):
            continue
        row = {'Колонка ods.scd1_z_r2_ip_merchants': c}
        for g in ('номер совпал', 'клиент совпал, номер нет'):
            m = s['group'] == g
            row['%s: точно' % g] = int(hit[m].sum())
            row['%s: по цифрам' % g] = int(hit_d[m].sum())
            row['%s: в выборке' % g] = int(m.sum())
        rows.append(row)
    cft_cols_t = pd.DataFrame(rows)
    if len(cft_cols_t):
        cft_cols_t = cft_cols_t.sort_values(['клиент совпал, номер нет: точно', 'номер совпал: точно'], ascending=False)
    display(cft_cols_t)

    best = CFT_NUM_COL
    if best is None and len(cft_cols_t):
        top = cft_cols_t.iloc[0]
        if top['клиент совпал, номер нет: точно'] + top['номер совпал: точно'] > 0:
            best = top['Колонка ods.scd1_z_r2_ip_merchants']
    print('Колонка с номером Снигура:', best)

    if best:
        ids = sorted(fd['id'].dropna().unique())
        parts = []
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            for i in range(0, len(ids), 1000):
                parts.append(imp.fetch(
                    "SELECT cast(m.id AS string) AS id, cast(m.%s AS string) AS cft_num "
                    "FROM ods.scd1_z_r2_ip_merchants m WHERE cast(m.id AS string) IN (%s) "
                    "AND coalesce(m.ods_deleted_flg, '0') <> '1'" % (best, sql_in(ids[i:i + 1000]))))
        cft = pd.concat(parts, ignore_index=True)
        cft['id'] = cft['id'].map(norm_id)
        cft['cft_key'] = cft['cft_num'].map(norm_agr)
        cft = cft.dropna(subset=['cft_key']).drop_duplicates('id').set_index('id')
        fd['Номер в ЦФТ (%s)' % best] = fd['id'].map(cft['cft_num'])
        fd['cft_key'] = fd['id'].map(cft['cft_key'])
        by_alpha = fd['num_key'].isin(sn_act_keys)
        by_cft = fd['cft_key'].isin(sn_act_keys)
        cft_match = pd.DataFrame([
            ('Договоров final_df за август', len(fd)),
            ('  номер ЦФТ найден', int(fd['cft_key'].notna().sum())),
            ('  номер ЦФТ отличается от номера ods_alpha', int((fd['cft_key'].notna() & (fd['cft_key'] != fd['num_key'])).sum())),
            ('Есть у Снигура (действующие) по номеру ods_alpha', int(by_alpha.sum())),
            ('Есть у Снигура (действующие) по номеру ЦФТ', int(by_cft.sum())),
            ('  только по номеру ЦФТ', int((by_cft & ~by_alpha).sum())),
            ('  только по номеру ods_alpha', int((by_alpha & ~by_cft).sum())),
            ('Есть у Снигура (действующие) хотя бы по одному', int((by_cft | by_alpha).sum())),
        ], columns=['Номер из ЦФТ, колонка %s' % best, 'Договоров'])
        display(cft_match)
        cft_examples = fd.loc[fd['cft_key'].notna() & (fd['cft_key'] != fd['num_key']),
                              ['agr_id (final_df)', 'Номер договора (final_df)', 'Номер в ЦФТ (%s)' % best,
                               'ИНН (final_df)', 'Наименование (final_df)']]
        display(cft_examples.head(15))
else:
    print('LOOKUP_CFT = False - пропущено')


## 6. Выгрузка

In [ ]:
def as_dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

def autofit(ws):
    for col in ws.columns:
        width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 10), 60)

cols_used = pd.DataFrame([
    ('ID договора', ex_id), ('Номер договора', ex_num), ('ИНН', ex_inn), ('Наименование', ex_name),
], columns=['Колонка отчёта', 'Взята'])

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:
    row = 0
    for block in [loaded, cols_used] + fd_rep_tables + hyp_tables + [cft_match]:
        if len(block):
            block.to_excel(writer, index=False, sheet_name='Сводка', startrow=row)
            row += len(block) + 2
    overlap.to_excel(writer, index=False, sheet_name='Совпадение ключей')
    as_dates(fd_rep_out).to_excel(writer, index=False, sheet_name='final_df и отчёт')
    as_dates(num_diff_out).to_excel(writer, index=False, sheet_name='Номер разный')
    as_dates(name_diff_out).to_excel(writer, index=False, sheet_name='Наименование разное')
    if len(cft_cols_t):
        cft_cols_t.to_excel(writer, index=False, sheet_name='ЦФТ - колонки')
    if len(cft_examples):
        cft_examples.to_excel(writer, index=False, sheet_name='ЦФТ - номер другой')
    for ws in writer.book.worksheets:
        autofit(ws)
        ws.freeze_panes = 'A2' if ws.title != 'Сводка' else None
print(OUT_FILE)
